In [1]:
import numpy as np
import warnings
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
from scipy.optimize import minimize
from scipy.stats import norm
warnings.filterwarnings('ignore')

In [2]:
data_in = np.load('initial_inputs.npy')
data_out = np.load('initial_outputs.npy')
print(data_in)
print(data_out)

[[0.19144708 0.03819337 0.60741781 0.41458414]
 [0.75865295 0.53651774 0.65600038 0.36034155]
 [0.43834987 0.8043397  0.21024527 0.15129482]
 [0.70605083 0.53419196 0.26424335 0.48208755]
 [0.83647799 0.19360965 0.6638927  0.78564888]
 [0.68343225 0.11866264 0.82904591 0.56757661]
 [0.55362148 0.66734998 0.32380582 0.81486975]
 [0.35235627 0.32224153 0.11697937 0.47311252]
 [0.15378571 0.72938169 0.42259844 0.44307417]
 [0.46344227 0.63002451 0.10790646 0.9576439 ]
 [0.67749115 0.35850951 0.47959222 0.07288048]
 [0.58397341 0.14724265 0.34809746 0.42861465]
 [0.30688872 0.31687813 0.62263448 0.09539906]
 [0.51114177 0.817957   0.72871042 0.11235362]
 [0.43893338 0.77409176 0.37816709 0.93369621]
 [0.22418902 0.84648049 0.87948418 0.87851568]
 [0.72526172 0.47987049 0.08894684 0.75976022]
 [0.35548161 0.63961937 0.41761768 0.12260384]
 [0.11987923 0.86254031 0.64333133 0.84980383]
 [0.12688467 0.15342962 0.77016219 0.19051811]]
[6.44434399e+01 1.83013796e+01 1.12939795e-01 4.21089813e+0

Week-01

In [3]:
X_init = np.array([
[0.19144708, 0.03819337, 0.60741781, 0.41458414],
[0.75865295, 0.53651774, 0.65600038, 0.36034155],
[0.43834987, 0.8043397 , 0.21024527, 0.15129482],
[0.70605083, 0.53419196, 0.26424335, 0.48208755],
[0.83647799, 0.19360965, 0.6638927 , 0.78564888],
[0.68343225, 0.11866264, 0.82904591, 0.56757661],
[0.55362148, 0.66734998, 0.32380582, 0.81486975],
[0.35235627, 0.32224153, 0.11697937, 0.47311252],
[0.15378571, 0.72938169, 0.42259844, 0.44307417],
[0.46344227, 0.63002451, 0.10790646, 0.9576439 ],
[0.67749115, 0.35850951, 0.47959222, 0.07288048],
[0.58397341, 0.14724265, 0.34809746, 0.42861465],
[0.30688872, 0.31687813, 0.62263448, 0.09539906],
[0.51114177, 0.817957  , 0.72871042, 0.11235362],
[0.43893338, 0.77409176, 0.37816709, 0.93369621],
[0.22418902, 0.84648049, 0.87948418, 0.87851568],
[0.72526172, 0.47987049, 0.08894684, 0.75976022],
[0.35548161, 0.63961937, 0.41761768, 0.12260384],
[0.11987923, 0.86254031, 0.64333133, 0.84980383],
[0.12688467, 0.15342962, 0.77016219, 0.19051811]
])

y_init = np.array([
6.44434399e+01, 1.83013796e+01, 1.12939795e-01, 4.21089813e+00,
2.58370525e+02, 7.84343889e+01, 5.75715369e+01, 1.09571876e+02,
8.84799176e+00, 2.33223610e+02, 2.44230883e+01, 6.44201468e+01,
6.34767158e+01, 7.97291299e+01, 3.55806818e+02, 1.08885962e+03,
2.88667516e+01, 4.51815703e+01, 4.31612757e+02, 9.97233189e+00
])

X_init = data_in
y_init = data_out


kernel = Matern(nu=2.5)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True)
gp.fit(X_init, y_init)

# Surrogate to maximise (negative for minimize)
def surrogate_neg(x):
    return -gp.predict(x.reshape(1, -1))[0]

# Bounds for normalized inputs
bounds = [(0,1), (0,1), (0,1), (0,1)]

# Try multiple random starts to avoid local issues
best_x = None
best_val = float('inf')
for _ in range(10):
    x0 = np.random.rand(4)
    res = minimize(surrogate_neg, x0=x0, bounds=bounds, method='L-BFGS-B')
    if res.fun < best_val:
        best_val = res.fun
        best_x = res.x

x_next = best_x
print("Next point to evaluate:", x_next)


Next point to evaluate: [0.03956228 0.28797962 0.07614284 0.92497613]


Week-02

In [4]:
X_init = data_in           # shape (n_samples, 4)
y_init = data_out          # shape (n_samples,)

# --- New evaluated point ---
x_new = np.array([0.232877,0.841416,0.883342,0.879464])
y_new = 1091.3271430129832

# --- Add new data to dataset ---
X_all = np.vstack([X_init, x_new])
y_all = np.hstack([y_init, y_new])

# --- Refit Gaussian Process ---
kernel = Matern(nu=2.5)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True)
gp.fit(X_all, y_all)

# --- Generate candidate next points around current best ---
best_x = x_new
sigma = 0.02  # tweak size for local exploration
num_candidates = 5

x_next_candidates = best_x + np.random.normal(0, sigma, size=(num_candidates, 4))
# Ensure all points are within [0,1]
x_next_candidates = np.clip(x_next_candidates, 0, 1)

print("Candidate next points to evaluate:")
print(x_next_candidates)

Candidate next points to evaluate:
[[0.26806048 0.83448079 0.89609099 0.86882406]
 [0.21713706 0.86913712 0.89414295 0.86439734]
 [0.23444426 0.84134702 0.91654077 0.86884979]
 [0.27897427 0.85515288 0.84659208 0.88512469]
 [0.21377101 0.82775101 0.90520325 0.87704095]]


Week-03

In [8]:
X_init = data_in           # shape (n_samples, 4)
y_init = data_out          # shape (n_samples,)

# --- Add the two new data points ---
new_points = np.array([
    [0.232877,0.841416,0.883342,0.879464],
    [0.245154,0.843081,0.898729,0.883391]
])

new_outputs = np.array([
    1091.3271430129832,
    1195.7279770056589
])


# Combine all data
X_all = np.vstack([X_init, new_points])
y_all = np.hstack([y_init, new_outputs])

# --- Fit updated Gaussian Process ---
kernel = Matern(nu=2.5)
gp = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-8,                # smaller noise term for precision
    normalize_y=True,
    n_restarts_optimizer=5     # more robust kernel fitting
)
gp.fit(X_all, y_all)

# --- Define acquisition function (UCB variant) ---
def surrogate_neg_ucb(x, kappa=2.0):
    mean, std = gp.predict(x.reshape(1, -1), return_std=True)
    return -(mean + kappa * std)

# --- Search bounds for normalized inputs ---
bounds = [(0, 1), (0, 1), (0, 1), (0, 1)]

# --- Optimize acquisition function for next sampling point ---
best_x, best_val = None, float('inf')
for _ in range(10):
    x0 = np.random.rand(4)
    res = minimize(surrogate_neg_ucb, x0=x0, bounds=bounds, method='L-BFGS-B')
    if res.fun < best_val:
        best_val = res.fun
        best_x = res.x

x_next = best_x

print("Suggested next point to evaluate:", x_next)

# --- Optionally: generate a few local perturbations for fine exploration ---
sigma = 0.01
num_candidates = 5
x_next_candidates = x_next + np.random.normal(0, sigma, size=(num_candidates, 4))
x_next_candidates = np.clip(x_next_candidates, 0, 1)

print("\nLocal candidate points for fine-tuning:")
print(x_next_candidates)

res_formatted = [f"{r:.6f}" for r in x_next]
result = "-".join(res_formatted)
print(result)

x_next_6dp = np.round(x_next, 6)
x_next_6dp

Suggested next point to evaluate: [0.23728611 0.89782819 0.94744474 0.89713363]

Local candidate points for fine-tuning:
[[0.25097428 0.89638504 0.93868086 0.90760148]
 [0.222098   0.89238205 0.95575257 0.88810064]
 [0.25487237 0.88804623 0.95256548 0.87514508]
 [0.21918576 0.89454904 0.95634564 0.8890761 ]
 [0.22402257 0.89681263 0.9395473  0.89216593]]
0.237286-0.897828-0.947445-0.897134


array([0.237286, 0.897828, 0.947445, 0.897134])